# ReLU on tensors of any shape

Element-wise kernels don't care about shape: an `(N, C, H, W)` activation in C-contiguous memory is just one long row of floats. Write `relu(x)` that applies

$$\mathrm{ReLU}(v) = \max(v, 0)$$

to every element of a `float32` array of **any** shape and returns a new `float32` array of the **same shape**.

## Requirements
- Flatten on the host (`x.ravel()` on a contiguous array is free: no copy), launch a 1-D kernel over all `x.size` elements, and reshape the result back to `x.shape`.
- Write the result **in place** in the device buffer: copy the input to the GPU once, let the kernel overwrite it, copy it back. No second device array is needed.
- Use 64 threads per block and $\lceil \text{size} / 64 \rceil$ blocks. An empty array (size 0) returns an empty array of the same shape without launching.
- `ReLU(-0.0)` is `0.0` or `-0.0`; either is accepted. Inputs contain no NaN.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda

TPB = 64

def relu(x):
    """x: float32 array of any shape. Returns max(x, 0) with the same shape."""
    # TODO: flatten, copy to the device, launch relu_kernel in place, copy back, reshape
    return x.copy()

@cuda.jit
def relu_kernel(buf):
    # TODO: one thread per element, overwrite buf[i] with max(buf[i], 0)
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [-2.0, -0.5, 0.0, 0.5, 2.0], "dtype": "float32"}], "e": {"__nd__": [0.0, 0.0, 0.0, 0.5, 2.0], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, -1.0], [-3.0, 4.0]], "dtype": "float32"}], "e": {"__nd__": [[1.0, 0.0], [0.0, 4.0]], "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0], [-1.0, -1.0, -1.0, -1.0, -1.0]], "dtype": "float32"}], "e": {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[2.040919065475464, -2.5556650161743164, 0.4180988371372223, -0.5677695870399475, -0.4526492953300476], [-0.21559716761112213, -2.019986152648926, -0.2319323718547821, -0.8652130961418152, 3.3229994773864746], [0.22578661143779755, -0.3526307940483093, -0.28128743171691895, -0.6680463552474976, -1.0551505088806152], [-0.39080098271369934, 0.4819453954696655, -0.23855361342430115, 0.9577587246894836, -0.19980213046073914]], [[0.02425956539809704, 1.5458208322525024, 0.545105516910553, -0.5052287578582764, -0.18283897638320923], [0.5405251383781433, 1.935088038444519, -0.2696203291416168, -0.24355867505073547, 1.0023136138916016], [-0.8864599466323853, -0.2917202413082123, 0.8825389742851257, 0.5803500413894653, 0.09151670336723328], [0.6701043844223022, -2.82816219329834, 1.0213068723678589, -0.9596447348594666, -1.6686198711395264]], [[0.27644574642181396, 0.7005448937416077, -0.44476744532585144, -1.0764058828353882, 0.026124833151698112], [-0.05274730920791626, 1.4055981636047363, 0.7474079728126526, 0.19381564855575562, 1.1116331815719604], [-0.2055230438709259, -0.9258995652198792, 0.5840582847595215, 0.5825384259223938, -0.21482890844345093], [-0.7828086018562317, 0.22915390133857727, -2.493894338607788, 0.6901247501373291, 0.49136826395988464]]], [[[-1.638857126235962, 0.06135350838303566, -0.964099645614624, 0.7572210431098938, -2.0341672897338867], [-0.914494514465332, 0.7095800042152405, 1.1564010381698608, -2.158005475997925, -0.4980398416519165], [0.3280200958251953, -0.6092161536216736, 1.5906401872634888, -1.1912267208099365, 0.3545319437980652], [-1.0484055280685425, 1.4059629440307617, -0.02165122888982296, -0.3722505569458008, -1.7181849479675293]], [[1.6818255186080933, 0.7527785897254944, 0.7535638213157654, 1.1378812789916992, 0.34922656416893005], [-0.6392465829849243, -0.800241231918335, -0.8001999855041504, 1.370072364807129, -1.4603811502456665], [-0.5963695049285889, -0.32124391198158264, 0.2246190309524536, 0.5753493905067444, -1.2490969896316528], [-1.7300134897232056, -0.004414232447743416, 1.2135637998580933, 0.7570580840110779, 0.2156507819890976]], [[-0.3171556293964386, 0.2932336926460266, -0.2433350831270218, 0.8172065615653992, -0.7944473624229431], [0.13423994183540344, -0.11078013479709625, 0.5433593988418579, 0.22463852167129517, 2.550034523010254], [1.4986547231674194, 1.4967371225357056, -2.039503812789917, -0.34031662344932556, -0.6086106300354004], [0.5327215790748596, -2.279026508331299, 1.1744986772537231, 1.0669833421707153, -1.3020708560943604]]]], "dtype": "float32"}], "e": {"__nd__": [[[[2.040919065475464, 0.0, 0.4180988371372223, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 3.3229994773864746], [0.22578661143779755, 0.0, 0.0, 0.0, 0.0], [0.0, 0.4819453954696655, 0.0, 0.9577587246894836, 0.0]], [[0.02425956539809704, 1.5458208322525024, 0.545105516910553, 0.0, 0.0], [0.5405251383781433, 1.935088038444519, 0.0, 0.0, 1.0023136138916016], [0.0, 0.0, 0.8825389742851257, 0.5803500413894653, 0.09151670336723328], [0.6701043844223022, 0.0, 1.0213068723678589, 0.0, 0.0]], [[0.27644574642181396, 0.7005448937416077, 0.0, 0.0, 0.026124833151698112], [0.0, 1.4055981636047363, 0.7474079728126526, 0.19381564855575562, 1.1116331815719604], [0.0, 0.0, 0.5840582847595215, 0.5825384259223938, 0.0], [0.0, 0.22915390133857727, 0.0, 0.6901247501373291, 0.49136826395988464]]], [[[0.0, 0.06135350838303566, 0.0, 0.7572210431098938, 0.0], [0.0, 0.7095800042152405, 1.1564010381698608, 0.0, 0.0], [0.3280200958251953, 0.0, 1.5906401872634888, 0.0, 0.3545319437980652], [0.0, 1.4059629440307617, 0.0, 0.0, 0.0]], [[1.6818255186080933, 0.7527785897254944, 0.7535638213157654, 1.1378812789916992, 0.34922656416893005], [0.0, 0.0, 0.0, 1.370072364807129, 0.0], [0.0, 0.0, 0.2246190309524536, 0.5753493905067444, 0.0], [0.0, 0.0, 1.2135637998580933, 0.7570580840110779, 0.2156507819890976]], [[0.0, 0.2932336926460266, 0.0, 0.8172065615653992, 0.0], [0.13423994183540344, 0.0, 0.5433593988418579, 0.22463852167129517, 2.550034523010254], [1.4986547231674194, 1.4967371225357056, 0.0, 0.0, 0.0], [0.5327215790748596, 0.0, 1.1744986772537231, 1.0669833421707153, 0.0]]]], "dtype": "float32"}}, {"a": [{"__nd__": [-0.978548526763916, -0.8011720180511475, 0.04329590126872063, 0.6409710645675659, 2.0478861331939697, -0.19744543731212616, 0.7675025463104248, 0.1554178148508072, 1.7599263191223145, 0.7421578764915466, 1.368550419807434, -1.077675223350525, -0.19224071502685547, -0.813772439956665, 1.5049474239349365, 0.6576399207115173, -0.30514442920684814, -0.4524678885936737, 0.4846648871898651, -0.7014955282211304, -0.9305889010429382, 0.48127448558807373, 2.46313214302063, -0.24613355100154877, -0.5558657646179199, -1.171156883239746, -1.3350110054016113, 0.5249832272529602, 0.850803017616272, 0.00917472131550312, 0.3325759768486023, 0.11591657996177673, 0.13865484297275543, -1.5261590480804443, -0.4581182599067688, 0.11147930473089218, -0.783166766166687, -0.4764297306537628, -0.8191201686859131, -0.33349665999412537, 0.853108286857605, -0.4065803587436676, -0.15387068688869476, 0.8137183785438538, 0.6447702050209045, 1.6952075958251953, -2.090485095977783, 0.8568589687347412, -0.4822840988636017, 0.13469181954860687, 0.8377193808555603, 1.0832531452178955, 1.0393507480621338, 0.15510669350624084, 1.609662652015686, -0.2829742133617401, -0.14098191261291504, 0.7993512153625488, -0.5513724088668823, 2.1609063148498535, 1.0192065238952637, 2.1755752563476562, -0.02658918686211109, -0.3830885589122772, 0.16704800724983215], "dtype": "float32"}], "e": {"__nd__": [0.0, 0.0, 0.04329590126872063, 0.6409710645675659, 2.0478861331939697, 0.0, 0.7675025463104248, 0.1554178148508072, 1.7599263191223145, 0.7421578764915466, 1.368550419807434, 0.0, 0.0, 0.0, 1.5049474239349365, 0.6576399207115173, 0.0, 0.0, 0.4846648871898651, 0.0, 0.0, 0.48127448558807373, 2.46313214302063, 0.0, 0.0, 0.0, 0.0, 0.5249832272529602, 0.850803017616272, 0.00917472131550312, 0.3325759768486023, 0.11591657996177673, 0.13865484297275543, 0.0, 0.0, 0.11147930473089218, 0.0, 0.0, 0.0, 0.0, 0.853108286857605, 0.0, 0.0, 0.8137183785438538, 0.6447702050209045, 1.6952075958251953, 0.0, 0.8568589687347412, 0.0, 0.13469181954860687, 0.8377193808555603, 1.0832531452178955, 1.0393507480621338, 0.15510669350624084, 1.609662652015686, 0.0, 0.0, 0.7993512153625488, 0.0, 2.1609063148498535, 1.0192065238952637, 2.1755752563476562, 0.0, 0.0, 0.16704800724983215], "dtype": "float32"}}, {"a": [{"__nd__": [[73.45764923095703, -58.742767333984375, 37.97017288208008, -1.6804280281066895, 161.56817626953125, -66.27017974853516, 104.61618041992188, -64.3863525390625, -96.06341552734375, -71.02994537353516, -119.01859283447266, 14.635035514831543, 103.12608337402344, 16.431488037109375, 62.43251037597656, 163.21742248535156, 27.002643585205078, 19.5162353515625, -27.506145477294922, -160.82423400878906, 75.97389221191406, -175.6460418701172, 65.26764678955078, -1.4320335388183594, 112.67814636230469, -6.7770256996154785, -82.32001495361328, 35.7910270690918, -56.023616790771484, -18.09568214416504, 4.187078952789307, -13.454970359802246, -18.88022804260254, -83.24449920654297, -18.91337776184082, -213.83407592773438, -15.733209609985352, -119.80559539794922, 112.0263671875, 126.99267578125], [-195.10256958007812, 14.491679191589355, -12.636391639709473, -104.67156219482422, 53.16531753540039, -46.16895294189453, -176.75991821289062, -26.667667388916016, -14.825311660766602, 10.643101692199707, -123.12329864501953, 61.568092346191406, 73.54570770263672, -114.58273315429688, -65.88555908203125, -8.033731460571289, -56.592529296875, 174.37545776367188, 20.83721160888672, -101.05821228027344, -78.81089782714844, -5.747025966644287, 229.577880859375, -17.826431274414062, 12.747642517089844, 51.404537200927734, -4.012593746185303, 228.0660858154297, -53.152496337890625, 74.42031860351562, 16.04379653930664, 62.1196403503418, -117.66429901123047, 176.81126403808594, -12.152750968933105, 3.4163262844085693, -93.49762725830078, -72.96952819824219, 56.42394256591797, 98.64999389648438], [75.32546997070312, 120.89703369140625, 71.44889068603516, 2.8461833000183105, 83.65259552001953, 59.36032485961914, -10.057526588439941, 72.60726928710938, 128.56973266601562, 23.45713233947754, -35.62051010131836, 71.87113952636719, 190.07217407226562, -21.05620574951172, -9.230962753295898, -13.639752388000488, 122.30174255371094, -183.71981811523438, 36.584537506103516, 119.22639465332031, -81.33831787109375, 148.89773559570312, 53.53090286254883, -54.90359115600586, 20.41398048400879, -153.44921875, -56.55710220336914, 182.6903839111328, -89.38040924072266, 184.45716857910156, -8.15735912322998, 99.4662857055664, 4.389702320098877, -224.1481475830078, -57.265132904052734, 18.98535919189453, -115.094970703125, -134.74107360839844, 36.61858367919922, -64.94149780273438], [-169.5827178955078, -68.41151428222656, 86.0737533569336, -47.37460708618164, 93.47725677490234, 152.72727966308594, 3.251539945602417, -112.96833801269531, -130.15650939941406, -2.6253855228424072, 88.1055679321289, 26.901535034179688, 54.7257194519043, -44.33417892456055, 30.04227638244629, -47.13087844848633, -32.56098937988281, -129.22463989257812, -157.22129821777344, -48.1956672668457, -110.75994110107422, -95.87457275390625, 67.10474395751953, -10.708091735839844, 291.41259765625, 92.63420867919922, -63.31875991821289, 77.80073547363281, 35.680389404296875, -68.56810760498047, 110.2594985961914, -77.53286743164062, -305.838134765625, 79.40647888183594, -61.725914001464844, 148.60374450683594, -64.96722412109375, -118.13542938232422, 154.84616088867188, -107.54983520507812], [20.339370727539062, 144.97271728515625, 11.44973373413086, -8.058491706848145, 10.24234390258789, 83.90536499023438, -80.68121337890625, 61.06410217285156, 57.707542419433594, 233.2711944580078, -33.65255355834961, -91.48298645019531, 73.5687026977539, -39.71572494506836, 28.911617279052734, 13.257652282714844, 116.83939361572266, -65.68374633789062, -9.22795295715332, -255.2112579345703, -89.01139068603516, 153.63388061523438, -68.15058898925781, -70.1476821899414, -121.79071807861328, 57.983428955078125, 17.79901695251465, -118.68669128417969, 66.74503326416016, 26.010587692260742, 75.40333557128906, -64.14794921875, 173.80087280273438, 40.98676300048828, -11.68895149230957, -51.88343048095703, -24.482736587524414, 85.87920379638672, -26.896671295166016, 107.81438446044922], [58.06761932373047, 134.64952087402344, 81.4104232788086, -6.293926239013672, -190.93399047851562, 29.857057571411133, 78.90760040283203, 16.289228439331055, 55.64316940307617, -47.162010192871094, -38.04775619506836, -94.23890686035156, 136.19004821777344, 15.839426040649414, -49.24860763549805, -133.1852569580078, -15.330727577209473, 144.26312255859375, 46.192161560058594, -72.45215606689453, 142.94949340820312, 142.85101318359375, 98.07762145996094, 78.88168334960938, 115.56517791748047, 54.32590103149414, 77.50008392333984, -78.00253295898438, 149.95578002929688, 23.118019104003906, 202.4211883544922, -204.2248077392578, 70.54088592529297, 98.14614868164062, -34.29326248168945, 45.24443054199219, 74.8326416015625, 57.7381477355957, -53.96592712402344, -14.1937837600708], [101.9039535522461, 127.73200225830078, 9.897684097290039, 7.703247547149658, 20.049636840820312, 103.76652526855469, -105.40385437011719, -132.90658569335938, 12.431496620178223, -110.65287780761719, -58.71094512939453, 8.629573822021484, 48.250492095947266, -107.20918273925781, 80.1605453491211, -177.3714599609375, 54.54160690307617, -157.77317810058594, -127.49234771728516, 64.2221908569336, -40.63622283935547, -79.55582427978516, -44.47767639160156, 46.50326156616211, -5.404679298400879, -2.93827748298645, 105.47764587402344, -71.6390609741211, -47.1933708190918, -52.24300003051758, 116.89221954345703, 99.14056396484375, 128.70765686035156, -3.375060796737671, -15.680254936218262, 85.27729797363281, -39.44116973876953, 83.06663513183594, 72.42927551269531, -69.83180236816406], [-8.319466590881348, -11.92509937286377, 173.33404541015625, 258.7823486328125, -9.992724418640137, 1.4064679145812988, -228.158447265625, 21.153228759765625, -0.7032972574234009, -107.28437805175781, 78.5594253540039, -156.57537841796875, -21.48529624938965, 13.885677337646484, -124.89999389648438, 85.674072265625, 105.5561752319336, -138.64208984375, -140.7838592529297, 60.22594451904297, -3.0497002601623535, -121.14031219482422, -29.522672653198242, -10.751667976379395, -14.303893089294434, 82.74886322021484, 38.41982650756836, 176.94638061523438, 137.43605041503906, 67.84737396240234, 107.70793914794922, 132.3819580078125, 70.5994873046875, -122.57421875, 85.94171142578125, 7.227258205413818, 161.881103515625, 9.347418785095215, 52.599246978759766, 32.9639892578125], [168.12255859375, 17.718231201171875, -26.953989028930664, 30.9091796875, -43.95082473754883, 16.20392417907715, -0.7472031116485596, 27.162256240844727, 192.32839965820312, -40.56905746459961, 3.6973209381103516, -19.949174880981445, 67.26115417480469, -118.15745544433594, -69.27294921875, -3.948960065841675, 54.614784240722656, -4.6281208992004395, -50.028629302978516, -86.6827392578125, 51.175262451171875, 144.21986389160156, 32.28755187988281, 18.81350326538086, 44.38730239868164, -94.26529693603516, -31.167490005493164, -26.79264259338379, 29.524045944213867, -3.202569007873535, -121.63999938964844, 112.52290344238281, 169.2408447265625, -26.319711685180664, 39.88408660888672, 26.716157913208008, 78.46636199951172, -147.97830200195312, 161.30096435546875, -46.78523254394531], [126.42262268066406, 8.06711483001709, -38.955562591552734, -14.045269966125488, 89.71729278564453, -86.3276138305664, -37.517791748046875, -48.21802520751953, -171.54483032226562, 130.0425262451172, 57.88888168334961, -2.131502151489258, 116.36626434326172, -95.64298248291016, -7.3449811935424805, 172.540283203125, 97.96664428710938, -222.5450439453125, -333.2081298828125, -103.79039001464844, 106.12833404541016, -47.24457931518555, -21.77440643310547, 60.82714080810547, -88.52022552490234, 21.287790298461914, 79.20982360839844, 46.70132064819336, 39.594547271728516, -60.204612731933594, 51.42871856689453, -42.76300811767578, 130.49468994140625, 257.00927734375, -73.99059295654297, -217.45021057128906, -42.54399108886719, 37.684513092041016, 105.01313018798828, 107.90155029296875], [-78.39513397216797, -79.32353973388672, 126.73009490966797, -109.23876953125, 109.02313232421875, -22.688793182373047, 76.6403579711914, 70.61647033691406, 77.66524505615234, 82.04267120361328, -98.93909454345703, 29.752111434936523, 1.3982411623001099, 30.840801239013672, 123.32483673095703, -27.110824584960938, -71.1970443725586, -72.8115463256836, -31.081836700439453, 75.26105499267578, -81.4815673828125, 115.92578887939453, 29.045146942138672, -22.61109161376953, -133.16510009765625, 24.493318557739258, 69.14693450927734, -176.5765838623047, 119.19596099853516, 164.7747039794922, 33.85022735595703, 49.1780891418457, 241.3695068359375, 123.18907165527344, -89.77571868896484, -1.6645185947418213, -15.693107604980469, -15.087678909301758, -236.42678833007812, 75.70048522949219], [-63.67361068725586, 16.36514663696289, -13.613934516906738, 87.10953521728516, -29.580869674682617, -79.54280090332031, 53.24224853515625, 140.4885711669922, -31.981069564819336, 136.9947967529297, 26.21831512451172, -53.27294921875, 152.51577758789062, -81.86263275146484, 101.40482330322266, -49.81859588623047, 45.02186584472656, -123.41482543945312, -83.75926208496094, 106.5052261352539, 18.81168556213379, 3.829301595687866, -43.9808235168457, 44.648067474365234, -139.7269287109375, -111.9935302734375, -15.779036521911621, -83.4281005859375, -70.05937957763672, -7.150325775146484, -13.752132415771484, -22.250669479370117, 166.52755737304688, 101.10758209228516, 122.97345733642578, -2.816200017929077, -50.46842575073242, -145.38990783691406, -34.81373977661133, -208.59556579589844], [-58.37413024902344, 76.42389678955078, 162.1530303955078, 96.5606918334961, -86.16004943847656, 49.2618522644043, -78.91639709472656, 82.29737854003906, -11.136117935180664, 85.82001495361328, 257.28955078125, 92.25243377685547, 233.2449951171875, 17.744232177734375, -32.971771240234375, 133.3704376220703, 172.43565368652344, -55.501461029052734, 7.663196563720703, -143.4529571533203, -20.004589080810547, 88.83322143554688, -8.638484001159668, -56.922019958496094, -140.7239990234375, 36.8254508972168, 79.75406646728516, 59.65007019042969, 25.53156852722168, -127.40653991699219, 176.1808624267578, -5.653636455535889, -14.479644775390625, 73.29285430908203, -216.31863403320312, -6.446723461151123, -13.678960800170898, 37.22831344604492, 134.7159881591797, 47.22001266479492], [-183.2098388671875, -33.907676696777344, 20.55316925048828, 14.716723442077637, -128.12628173828125, -166.3758544921875, 54.90108871459961, 34.35350799560547, -2.1301069259643555, -172.54220581054688, 177.405029296875, -290.02337646484375, -36.40972900390625, 209.11514282226562, -15.742586135864258, 23.207021713256836, -117.91464233398438, 135.864013671875, -129.26229858398438, 25.441917419433594, 13.490565299987793, -105.07244873046875, 88.69793701171875, -141.59580993652344, -68.34845733642578, -2.870220422744751, -41.24977111816406, 46.78066635131836, 92.40548706054688, -29.377376556396484, -54.1465950012207, -51.697818756103516, -36.55939483642578, 80.96488952636719, -21.758501052856445, 40.763954162597656, -9.115278244018555, -84.44651794433594, 86.37197875976562, -103.80412292480469], [0.619779646396637, 148.70237731933594, -109.31069946289062, -5.518852233886719, -158.4781036376953, -50.941627502441406, -149.45997619628906, -36.15871810913086, -53.82974624633789, 13.197161674499512, 158.43008422851562, 28.298452377319336, 65.81744384765625, -5.070193767547607, -24.452362060546875, -83.39817810058594, -180.08567810058594, 6.86203670501709, 162.912353515625, 47.62998962402344, -53.398895263671875, 164.94061279296875, -43.262351989746094, -61.88397216796875, -121.26943969726562, -96.20509338378906, -24.63628578186035, 199.90206909179688, -20.61762046813965, -103.1516342163086, -60.54186248779297, -22.286176681518555, -171.17520141601562, 12.662842750549316, 70.06684875488281, 32.08122634887695, 60.07326126098633, -128.9547119140625, -169.85919189453125, 128.5574493408203], [109.43313598632812, -154.0264434814453, 20.638166427612305, 2.5527870655059814, 54.48862838745117, 20.85862159729004, -50.21318817138672, 19.525663375854492, 15.70468807220459, -4.862762928009033, -185.2418670654297, -299.5201110839844, -79.01887512207031, -163.47714233398438, 25.25620460510254, 71.60967254638672, 26.31154441833496, -36.952789306640625, 7.048084735870361, 116.73410034179688, -72.88080596923828, 92.22947692871094, 70.76163482666016, 26.168426513671875, 145.19302368164062, -71.78185272216797, 94.25060272216797, -92.89515686035156, -76.70315551757812, -105.97110748291016, 44.583160400390625, -93.97600555419922, 5.215989589691162, -113.67119598388672, 106.62576293945312, 183.6173553466797, 22.607664108276367, 164.33653259277344, 56.06636428833008, 216.93101501464844]], "dtype": "float32"}], "e": {"__nd__": [[73.45764923095703, 0.0, 37.97017288208008, 0.0, 161.56817626953125, 0.0, 104.61618041992188, 0.0, 0.0, 0.0, 0.0, 14.635035514831543, 103.12608337402344, 16.431488037109375, 62.43251037597656, 163.21742248535156, 27.002643585205078, 19.5162353515625, 0.0, 0.0, 75.97389221191406, 0.0, 65.26764678955078, 0.0, 112.67814636230469, 0.0, 0.0, 35.7910270690918, 0.0, 0.0, 4.187078952789307, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 112.0263671875, 126.99267578125], [0.0, 14.491679191589355, 0.0, 0.0, 53.16531753540039, 0.0, 0.0, 0.0, 0.0, 10.643101692199707, 0.0, 61.568092346191406, 73.54570770263672, 0.0, 0.0, 0.0, 0.0, 174.37545776367188, 20.83721160888672, 0.0, 0.0, 0.0, 229.577880859375, 0.0, 12.747642517089844, 51.404537200927734, 0.0, 228.0660858154297, 0.0, 74.42031860351562, 16.04379653930664, 62.1196403503418, 0.0, 176.81126403808594, 0.0, 3.4163262844085693, 0.0, 0.0, 56.42394256591797, 98.64999389648438], [75.32546997070312, 120.89703369140625, 71.44889068603516, 2.8461833000183105, 83.65259552001953, 59.36032485961914, 0.0, 72.60726928710938, 128.56973266601562, 23.45713233947754, 0.0, 71.87113952636719, 190.07217407226562, 0.0, 0.0, 0.0, 122.30174255371094, 0.0, 36.584537506103516, 119.22639465332031, 0.0, 148.89773559570312, 53.53090286254883, 0.0, 20.41398048400879, 0.0, 0.0, 182.6903839111328, 0.0, 184.45716857910156, 0.0, 99.4662857055664, 4.389702320098877, 0.0, 0.0, 18.98535919189453, 0.0, 0.0, 36.61858367919922, 0.0], [0.0, 0.0, 86.0737533569336, 0.0, 93.47725677490234, 152.72727966308594, 3.251539945602417, 0.0, 0.0, 0.0, 88.1055679321289, 26.901535034179688, 54.7257194519043, 0.0, 30.04227638244629, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 67.10474395751953, 0.0, 291.41259765625, 92.63420867919922, 0.0, 77.80073547363281, 35.680389404296875, 0.0, 110.2594985961914, 0.0, 0.0, 79.40647888183594, 0.0, 148.60374450683594, 0.0, 0.0, 154.84616088867188, 0.0], [20.339370727539062, 144.97271728515625, 11.44973373413086, 0.0, 10.24234390258789, 83.90536499023438, 0.0, 61.06410217285156, 57.707542419433594, 233.2711944580078, 0.0, 0.0, 73.5687026977539, 0.0, 28.911617279052734, 13.257652282714844, 116.83939361572266, 0.0, 0.0, 0.0, 0.0, 153.63388061523438, 0.0, 0.0, 0.0, 57.983428955078125, 17.79901695251465, 0.0, 66.74503326416016, 26.010587692260742, 75.40333557128906, 0.0, 173.80087280273438, 40.98676300048828, 0.0, 0.0, 0.0, 85.87920379638672, 0.0, 107.81438446044922], [58.06761932373047, 134.64952087402344, 81.4104232788086, 0.0, 0.0, 29.857057571411133, 78.90760040283203, 16.289228439331055, 55.64316940307617, 0.0, 0.0, 0.0, 136.19004821777344, 15.839426040649414, 0.0, 0.0, 0.0, 144.26312255859375, 46.192161560058594, 0.0, 142.94949340820312, 142.85101318359375, 98.07762145996094, 78.88168334960938, 115.56517791748047, 54.32590103149414, 77.50008392333984, 0.0, 149.95578002929688, 23.118019104003906, 202.4211883544922, 0.0, 70.54088592529297, 98.14614868164062, 0.0, 45.24443054199219, 74.8326416015625, 57.7381477355957, 0.0, 0.0], [101.9039535522461, 127.73200225830078, 9.897684097290039, 7.703247547149658, 20.049636840820312, 103.76652526855469, 0.0, 0.0, 12.431496620178223, 0.0, 0.0, 8.629573822021484, 48.250492095947266, 0.0, 80.1605453491211, 0.0, 54.54160690307617, 0.0, 0.0, 64.2221908569336, 0.0, 0.0, 0.0, 46.50326156616211, 0.0, 0.0, 105.47764587402344, 0.0, 0.0, 0.0, 116.89221954345703, 99.14056396484375, 128.70765686035156, 0.0, 0.0, 85.27729797363281, 0.0, 83.06663513183594, 72.42927551269531, 0.0], [0.0, 0.0, 173.33404541015625, 258.7823486328125, 0.0, 1.4064679145812988, 0.0, 21.153228759765625, 0.0, 0.0, 78.5594253540039, 0.0, 0.0, 13.885677337646484, 0.0, 85.674072265625, 105.5561752319336, 0.0, 0.0, 60.22594451904297, 0.0, 0.0, 0.0, 0.0, 0.0, 82.74886322021484, 38.41982650756836, 176.94638061523438, 137.43605041503906, 67.84737396240234, 107.70793914794922, 132.3819580078125, 70.5994873046875, 0.0, 85.94171142578125, 7.227258205413818, 161.881103515625, 9.347418785095215, 52.599246978759766, 32.9639892578125], [168.12255859375, 17.718231201171875, 0.0, 30.9091796875, 0.0, 16.20392417907715, 0.0, 27.162256240844727, 192.32839965820312, 0.0, 3.6973209381103516, 0.0, 67.26115417480469, 0.0, 0.0, 0.0, 54.614784240722656, 0.0, 0.0, 0.0, 51.175262451171875, 144.21986389160156, 32.28755187988281, 18.81350326538086, 44.38730239868164, 0.0, 0.0, 0.0, 29.524045944213867, 0.0, 0.0, 112.52290344238281, 169.2408447265625, 0.0, 39.88408660888672, 26.716157913208008, 78.46636199951172, 0.0, 161.30096435546875, 0.0], [126.42262268066406, 8.06711483001709, 0.0, 0.0, 89.71729278564453, 0.0, 0.0, 0.0, 0.0, 130.0425262451172, 57.88888168334961, 0.0, 116.36626434326172, 0.0, 0.0, 172.540283203125, 97.96664428710938, 0.0, 0.0, 0.0, 106.12833404541016, 0.0, 0.0, 60.82714080810547, 0.0, 21.287790298461914, 79.20982360839844, 46.70132064819336, 39.594547271728516, 0.0, 51.42871856689453, 0.0, 130.49468994140625, 257.00927734375, 0.0, 0.0, 0.0, 37.684513092041016, 105.01313018798828, 107.90155029296875], [0.0, 0.0, 126.73009490966797, 0.0, 109.02313232421875, 0.0, 76.6403579711914, 70.61647033691406, 77.66524505615234, 82.04267120361328, 0.0, 29.752111434936523, 1.3982411623001099, 30.840801239013672, 123.32483673095703, 0.0, 0.0, 0.0, 0.0, 75.26105499267578, 0.0, 115.92578887939453, 29.045146942138672, 0.0, 0.0, 24.493318557739258, 69.14693450927734, 0.0, 119.19596099853516, 164.7747039794922, 33.85022735595703, 49.1780891418457, 241.3695068359375, 123.18907165527344, 0.0, 0.0, 0.0, 0.0, 0.0, 75.70048522949219], [0.0, 16.36514663696289, 0.0, 87.10953521728516, 0.0, 0.0, 53.24224853515625, 140.4885711669922, 0.0, 136.9947967529297, 26.21831512451172, 0.0, 152.51577758789062, 0.0, 101.40482330322266, 0.0, 45.02186584472656, 0.0, 0.0, 106.5052261352539, 18.81168556213379, 3.829301595687866, 0.0, 44.648067474365234, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 166.52755737304688, 101.10758209228516, 122.97345733642578, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 76.42389678955078, 162.1530303955078, 96.5606918334961, 0.0, 49.2618522644043, 0.0, 82.29737854003906, 0.0, 85.82001495361328, 257.28955078125, 92.25243377685547, 233.2449951171875, 17.744232177734375, 0.0, 133.3704376220703, 172.43565368652344, 0.0, 7.663196563720703, 0.0, 0.0, 88.83322143554688, 0.0, 0.0, 0.0, 36.8254508972168, 79.75406646728516, 59.65007019042969, 25.53156852722168, 0.0, 176.1808624267578, 0.0, 0.0, 73.29285430908203, 0.0, 0.0, 0.0, 37.22831344604492, 134.7159881591797, 47.22001266479492], [0.0, 0.0, 20.55316925048828, 14.716723442077637, 0.0, 0.0, 54.90108871459961, 34.35350799560547, 0.0, 0.0, 177.405029296875, 0.0, 0.0, 209.11514282226562, 0.0, 23.207021713256836, 0.0, 135.864013671875, 0.0, 25.441917419433594, 13.490565299987793, 0.0, 88.69793701171875, 0.0, 0.0, 0.0, 0.0, 46.78066635131836, 92.40548706054688, 0.0, 0.0, 0.0, 0.0, 80.96488952636719, 0.0, 40.763954162597656, 0.0, 0.0, 86.37197875976562, 0.0], [0.619779646396637, 148.70237731933594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 13.197161674499512, 158.43008422851562, 28.298452377319336, 65.81744384765625, 0.0, 0.0, 0.0, 0.0, 6.86203670501709, 162.912353515625, 47.62998962402344, 0.0, 164.94061279296875, 0.0, 0.0, 0.0, 0.0, 0.0, 199.90206909179688, 0.0, 0.0, 0.0, 0.0, 0.0, 12.662842750549316, 70.06684875488281, 32.08122634887695, 60.07326126098633, 0.0, 0.0, 128.5574493408203], [109.43313598632812, 0.0, 20.638166427612305, 2.5527870655059814, 54.48862838745117, 20.85862159729004, 0.0, 19.525663375854492, 15.70468807220459, 0.0, 0.0, 0.0, 0.0, 0.0, 25.25620460510254, 71.60967254638672, 26.31154441833496, 0.0, 7.048084735870361, 116.73410034179688, 0.0, 92.22947692871094, 70.76163482666016, 26.168426513671875, 145.19302368164062, 0.0, 94.25060272216797, 0.0, 0.0, 0.0, 44.583160400390625, 0.0, 5.215989589691162, 0.0, 106.62576293945312, 183.6173553466797, 22.607664108276367, 164.33653259277344, 56.06636428833008, 216.93101501464844]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(relu, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")